# Pneumonia detection from chest X-rays with a CNN

**Task:** classify a chest X-ray as *normal* or *pneumonia*.

**Why a CNN:** pneumonia shows up as local patterns (cloudy white areas) inside the lungs. Convolution filters learn such local patterns directly from the pixels and pooling makes them robust to small shifts, so no hand-made features are needed.

**Why recall matters:** a missed pneumonia (false negative) is far worse than a false alarm, so besides accuracy we report recall for each class.

| Step | Section |
|---|---|
| Dataset | 1 |
| Preprocess | 2 |
| CNN model | 3 |
| Train | 4 |
| Save | 5 |
| Test | 6 |
| Understand | 4, 7 |
| Predict | 8 |

In [ ]:
import io
import json
import math
import time
import urllib.request
import zipfile
from pathlib import Path, PurePosixPath

import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import torch
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
MODEL_PATH = OUTPUT_DIR / "model.pt"
SAMPLES_DIR = ROOT / "samples"
OUTPUT_DIR.mkdir(exist_ok=True)

IMG_SIZE = 128
CLASSES = ["normal", "pneumonia"]
EPOCHS = 15
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
SEED = 0

torch.manual_seed(SEED)

## 1. Dataset

The images are adult chest X-rays from the **RSNA Pneumonia Detection Challenge** (radiologist-labelled, originally from the NIH ChestX-ray collection). We take them from the [COVID-19 Radiography Database](https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database) on Kaggle, which provides them as 299×299 PNGs. Download size is 816 MB, and no login is needed.

- **pneumonia:** all 6,012 *Lung Opacity* images, which are opacities suggestive of pneumonia.
- **normal:** 6,012 images randomly sampled from the 8,851 RSNA *Normal* images.
- We use **only RSNA images**. If the two classes came from different hospitals, the CNN could learn to recognise the hospital instead of the disease. The metadata sheets tell us the source of every image.
- The 12,024 images are resized to 128×128 and split 70/15/15, stratified with a fixed seed: **train 8,416 / validation 1,804 / test 1,804, exactly 50/50 in every split**.

In [ ]:
SOURCE_FOLDERS = ["Normal", "Lung_Opacity"]
ARCHIVE = ROOT / "data" / "raw" / "covid19-radiography-database.zip"
ARCHIVE_URL = "https://www.kaggle.com/api/v1/datasets/download/tawsifurrahman/covid19-radiography-database"
ARCHIVE_ROOT = "COVID-19_Radiography_Dataset"
RSNA_URL = "https://www.kaggle.com/c/rsna-pneumonia-detection-challenge/data"
DATA_FILE = ROOT / "data" / f"rsna_pneumonia_{IMG_SIZE}.npz"


def download():
    if ARCHIVE.exists():
        return
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    partial = ARCHIVE.with_suffix(".part")
    print(f"Downloading {ARCHIVE.name} (816 MB) ...")
    urllib.request.urlretrieve(ARCHIVE_URL, partial)
    partial.rename(ARCHIVE)


def fit_square(image):
    return ImageOps.fit(image.convert("L"), (IMG_SIZE, IMG_SIZE), Image.BICUBIC)


def rsna_images(archive, folder):
    prefix = f"{ARCHIVE_ROOT}/{folder}/images/"
    files = {PurePosixPath(name).stem.lower(): name for name in archive.namelist() if name.startswith(prefix)}
    metadata = io.BytesIO(archive.read(f"{ARCHIVE_ROOT}/{folder}.metadata.xlsx"))
    sheet = openpyxl.load_workbook(metadata, read_only=True).active
    rows = sheet.iter_rows(min_row=2, values_only=True)
    return sorted(files[name.lower()] for name, *_, url in rows if url == RSNA_URL)


def build():
    download()
    rng = np.random.default_rng(SEED)
    with zipfile.ZipFile(ARCHIVE) as archive:
        normal, pneumonia = (rsna_images(archive, folder) for folder in SOURCE_FOLDERS)
        normal = sorted(rng.choice(normal, size=len(pneumonia), replace=False).tolist())
        paths = normal + pneumonia
        print(f"Resizing {len(paths)} images to {IMG_SIZE}x{IMG_SIZE} ...")
        images = np.stack([np.asarray(fit_square(Image.open(archive.open(path)))) for path in paths])
    labels = np.array([0] * len(normal) + [1] * len(pneumonia), dtype=np.uint8)

    indices = np.arange(len(labels))
    train, rest = train_test_split(indices, test_size=0.3, stratify=labels, random_state=SEED)
    val, test = train_test_split(rest, test_size=0.5, stratify=labels[rest], random_state=SEED)
    splits = {"train": train, "val": val, "test": test}
    np.savez(
        DATA_FILE,
        **{f"{split}_images": images[i] for split, i in splits.items()},
        **{f"{split}_labels": labels[i] for split, i in splits.items()},
    )


def load_split(split):
    with np.load(DATA_FILE) as data:
        return data[f"{split}_images"], data[f"{split}_labels"]


if not DATA_FILE.exists():
    build()
for split in ("train", "val", "test"):
    _, labels = load_split(split)
    counts = ", ".join(f"{name} {np.sum(labels == i)}" for i, name in enumerate(CLASSES))
    print(f"{split:<6}{len(labels):>6} images  ({counts})")

In [ ]:
images, labels = load_split("train")
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for row, (label, name) in zip(axes, enumerate(CLASSES)):
    for ax, i in zip(row, np.flatnonzero(labels == label)[:6]):
        ax.imshow(images[i], cmap="gray")
        ax.set_title(name, fontsize=9)
        ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "dataset_samples.png", dpi=150)
plt.show()

## 2. Preprocessing

- Pixel values are scaled from 0–255 to 0–1.
- **Augmentation**, applied to training images only: random rotation of ±10°, shift of ±5% and zoom of ±5%. This helps against overfitting. There are no flips, because left and right matter in chest anatomy (for example, the heart is on the left).
- The classes are balanced (50/50), so a plain cross-entropy loss is enough and accuracy is meaningful: a model that always answers "pneumonia" would score only 50%.

In [ ]:
AUGMENT = v2.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05))


class XRayDataset(Dataset):
    def __init__(self, split, augment=False):
        images, labels = load_split(split)
        self.images = torch.from_numpy(images).unsqueeze(1).float() / 255
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        image = self.images[i]
        if self.augment:
            image = AUGMENT(image)
        return image, self.labels[i]


def preprocess_image(path):
    image = fit_square(Image.open(path))
    return torch.from_numpy(np.asarray(image, dtype=np.float32) / 255).unsqueeze(0)


train_set = XRayDataset("train", augment=True)
val_set = XRayDataset("val")
test_set = XRayDataset("test")

original = train_set.images[0]
fig, axes = plt.subplots(1, 5, figsize=(11, 2.6))
axes[0].imshow(original[0], cmap="gray")
axes[0].set_title("original", fontsize=9)
for ax in axes[1:]:
    ax.imshow(AUGMENT(original)[0], cmap="gray")
    ax.set_title("augmented", fontsize=9)
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "augmentation.png", dpi=150)
plt.show()

## 3. CNN model

- **Feature extractor:** 4 conv blocks, each *Conv 3×3 → BatchNorm → ReLU → MaxPool 2×2*. The number of filters grows 16 → 32 → 64 → 128 while the image shrinks 128 → 64 → 32 → 16 → 8.
- **Classifier:** *Flatten → Dropout 0.5 → Dense 128 → ReLU → Dropout 0.5 → Dense 2 → Softmax*. This turns the features into class probabilities.

The softmax is part of the loss function (`CrossEntropyLoss`) during training. At prediction time it is applied explicitly.

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 16),
            conv_block(16, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128 * (IMG_SIZE // 16) ** 2, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, len(CLASSES)),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = PneumoniaCNN()

In [ ]:
DIAGRAM_LAYERS = {
    nn.Conv2d: ("Conv 3x3", "#f0a35e"),
    nn.MaxPool2d: ("Max pool", "#f6cfa5"),
    nn.Flatten: ("Flatten", "#3b4a6b"),
    nn.Linear: ("Dense", "#3b4a6b"),
}


def layer_outputs(model):
    rows = []
    leaves = [module for module in model.modules() if not list(module.children())]
    hooks = [m.register_forward_hook(lambda m, _, out: rows.append((m, tuple(out.shape[1:])))) for m in leaves]
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, 1, IMG_SIZE, IMG_SIZE))
    for hook in hooks:
        hook.remove()
    return rows


def write_summary(rows):
    lines = [f"{'Layer':<14}{'Output shape':<20}{'Params':>10}"]
    for module, shape in rows:
        params = sum(p.numel() for p in module.parameters())
        lines.append(f"{type(module).__name__:<14}{' x '.join(map(str, shape)):<20}{params:>10,}")
    total = sum(sum(p.numel() for p in module.parameters()) for module, _ in rows)
    lines.append(f"Total parameters: {total:,}")
    summary = "\n".join(lines)
    print(summary)
    (OUTPUT_DIR / "model_summary.txt").write_text(summary)


def box_size(shape):
    if len(shape) == 3:
        return 0.25 + 0.08 * math.log2(shape[0]), 0.4 + 3.0 * shape[1] / IMG_SIZE
    return 0.2, 0.3 + 0.35 * math.log10(shape[0])


def plot_structure(rows):
    steps = [("Input", "#d9d2c5", (1, IMG_SIZE, IMG_SIZE))]
    steps += [(*DIAGRAM_LAYERS[type(m)], shape) for m, shape in rows if type(m) in DIAGRAM_LAYERS]
    steps.append(("Softmax", "#1f2a44", rows[-1][1]))

    fig, ax = plt.subplots(figsize=(1.25 * len(steps), 4.2))
    for i, (name, color, shape) in enumerate(steps):
        width, height = box_size(shape)
        ax.add_patch(Rectangle((i - width / 2, -height / 2), width, height, color=color))
        ax.text(i, -1.85, name, ha="center", va="top", fontsize=9, fontweight="bold")
        ax.text(i, -2.2, "x".join(map(str, shape)), ha="center", va="top", fontsize=8, color="#555")
    ax.set(xlim=(-0.7, len(steps) - 0.3), ylim=(-2.6, 1.9), title="CNN structure (output shape of each step)")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "structure_diagram.png", dpi=150)
    plt.show()


rows = layer_outputs(model)
write_summary(rows)
plot_structure(rows)

## 4. Training

- Optimizer: Adam (learning rate 0.001), batch size 64, 15 epochs, cross-entropy loss.
- After every epoch the model is checked on the validation set. The weights are saved whenever validation accuracy improves.
- Expect about 10 minutes on a CPU (about 40 s per epoch).

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct = 0.0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            logits = model(images)
            loss = loss_fn(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = {key: [] for key in ("train_loss", "train_acc", "val_loss", "val_acc")}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    results = (*run_epoch(model, train_loader, loss_fn, optimizer), *run_epoch(model, val_loader, loss_fn))
    for key, value in zip(history, results):
        history[key].append(value)
    train_loss, train_acc, val_loss, val_acc = results
    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
    print(
        f"epoch {epoch:2d}/{EPOCHS}  "
        f"train loss {train_loss:.3f} acc {train_acc:.3f}  "
        f"val loss {val_loss:.3f} acc {val_acc:.3f}  "
        f"{time.time() - start:.0f}s{'  saved' if improved else ''}"
    )

(OUTPUT_DIR / "history.json").write_text(json.dumps(history, indent=2))
print(f"Best validation accuracy {best_val_acc:.3f}")

### Learning curves

If the training and validation curves stay close together, the model generalises. If training keeps improving while validation gets worse, the model is overfitting.

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in zip(axes, ("loss", "acc"), ("Loss", "Accuracy")):
    for split in ("train", "val"):
        ax.plot(epochs, history[f"{split}_{metric}"], marker="o", markersize=3, label=split)
    ax.set(title=title, xlabel="Epoch")
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "learning_curves.png", dpi=150)
plt.show()

## 5. Save and reload

The best weights are stored in `outputs/model.pt`. Reloading them into a fresh model is exactly what happens when the model is used later on new images.

In [ ]:
model = PneumoniaCNN()
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu", weights_only=True))
model.eval()
print(f"Loaded {MODEL_PATH.relative_to(ROOT)} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)")

## 6. Testing

The test set has 1,804 images (902 per class) the model has never seen.
- **Recall (pneumonia):** the share of sick patients the model finds.
- **Recall (normal):** the share of healthy patients correctly cleared.

In [ ]:
with torch.no_grad():
    logits = torch.cat([model(batch) for batch in test_set.images.split(256)])
y_true = test_set.labels.numpy()
y_pred = logits.argmax(1).numpy()

report = classification_report(y_true, y_pred, target_names=CLASSES, digits=3)
summary = f"Test accuracy: {accuracy_score(y_true, y_pred):.3f}\n\n{report}"
print(summary)
(OUTPUT_DIR / "test_metrics.txt").write_text(summary)

matrix = ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=CLASSES, cmap="Blues", colorbar=False)
matrix.ax_.set(title="Confusion matrix (test set)", xlabel="Predicted", ylabel="True")
matrix.figure_.tight_layout()
matrix.figure_.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

## 7. Feature maps

These images show what each conv block "sees": the first 8 activation maps for one normal and one pneumonia test image. Early layers respond to edges and ribs. Deeper layers are coarser and more abstract.

In [ ]:
def plot_feature_maps(model, image, name, channels=8):
    activations = []
    relus = [m for m in model.features.modules() if isinstance(m, nn.ReLU)]
    hooks = [m.register_forward_hook(lambda _, __, out: activations.append(out[0])) for m in relus]
    with torch.no_grad():
        probs = model(image.unsqueeze(0)).softmax(1)[0]
    for hook in hooks:
        hook.remove()

    fig, axes = plt.subplots(len(activations) + 1, channels, figsize=(channels * 1.4, (len(activations) + 1) * 1.5))
    for ax in axes.flat:
        ax.axis("off")
    axes[0, 0].imshow(image[0], cmap="gray")
    axes[0, 0].set_title("input", fontsize=8)
    for row, fmap in zip(axes[1:], activations):
        for c, ax in enumerate(row):
            ax.imshow(fmap[c], cmap="viridis")
        row[0].set_title(f"conv {fmap.shape[0]}x{fmap.shape[1]}x{fmap.shape[2]}", fontsize=8, loc="left")
    prediction = CLASSES[probs.argmax().item()]
    fig.suptitle(f"Feature maps - true: {name}, predicted: {prediction} ({probs.max().item():.1%})")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"feature_maps_{name}.png", dpi=150)
    plt.show()


for label, name in enumerate(CLASSES):
    index = int((test_set.labels == label).nonzero()[0])
    plot_feature_maps(model, test_set.images[index], name)

## 8. Predict new images

`samples/` holds test images saved as PNG files. To classify any other chest X-ray, drop a PNG or JPG into that folder and run this cell again. Each image is converted to grayscale, center-cropped and resized to 128×128, the same way as the dataset.

In [ ]:
paths = [p for p in sorted(SAMPLES_DIR.iterdir()) if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
cols = 5
grid_rows = math.ceil(len(paths) / cols)
fig, axes = plt.subplots(grid_rows, cols, figsize=(cols * 2.6, grid_rows * 2.9), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, path in zip(axes.flat, paths):
    image = preprocess_image(path)
    with torch.no_grad():
        probs = model(image.unsqueeze(0)).softmax(1)[0]
    best = probs.argmax().item()
    details = "  ".join(f"{name} {p:.1%}" for name, p in zip(CLASSES, probs.tolist()))
    print(f"{path.name}: {CLASSES[best].upper()} ({probs[best].item():.1%})   [{details}]")
    ax.imshow(image[0], cmap="gray")
    ax.set_title(f"{path.stem}\n-> {CLASSES[best]} {probs[best].item():.0%}", fontsize=8)
fig.tight_layout()
plt.show()